In [ ]:
import chromadb
from datetime import datetime

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="audio_collection",
    metadata={
        "description": "A collection of audio data",
        "created": str(datetime.now())
    }
    )

In [9]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
df = pd.read_csv("hf://datasets/Aashish17405/my-audio-dataset/audio_dataset.csv")

print(df.columns.tolist()); print(df.head(2))

['audio', 'transcript']
              audio                    transcript
0  audio_file_1.wav             my name is Rocky.
1  audio_file_2.wav  Hello world, this is a test.


In [10]:
collection.add(
    ids=[str(i) for i in range(len(df))],
    documents=df["transcript"].tolist(),
    metadatas=[{"audio_path": row["audio"]} for _, row in df.iterrows()]
)

print(f"Added {collection.count()} documents")


Added 3 documents


In [11]:
results = collection.query(
    query_texts=["what is your name?"],
    n_results=2
)
print(results)

{'ids': [['0', '1']], 'embeddings': None, 'documents': [['my name is Rocky.', 'Hello world, this is a test.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'audio_path': 'audio_file_1.wav'}, {'audio_path': 'audio_file_2.wav'}]], 'distances': [[0.910591185092926, 1.536063551902771]]}


In [12]:
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display

# Get the best match
best = results["metadatas"][0][0]["audio_path"]

# Download from HuggingFace
local_path = hf_hub_download(
    repo_id="Aashish17405/my-audio-dataset",
    filename=best,
    repo_type="dataset"
)

display(Audio(local_path))


RemoteEntryNotFoundError: 404 Client Error. (Request ID: Root=1-69b258e2-006c4547187b998272198b34;0f377a4d-bc7d-43ec-9148-e58e64cd801e)

Entry Not Found for url: https://huggingface.co/datasets/Aashish17405/my-audio-dataset/resolve/main/audio_file_1.wav.

This dataset is incomplete and does not actually have audio files uploaded :'(